# Web Search Tool with Anthropic

This notebook demonstrates how to configure and use the built-in web search tool with Anthropic.

It covers:
- Environment and model setup
- Conversation helper functions
- Web search tool configuration
- A practical search example

## Environment Setup

Load environment variables, initialize the Anthropic client, and choose the model for web-enabled responses.

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

## Helper Functions

These helpers keep message history consistent and provide a single function for model calls with optional tools.

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

## Web Search Tool Configuration

This schema enables web search with configurable limits and domain restrictions for higher precision.

In [3]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

## Example Query

This final example sends a real question and allows the model to use the configured web search tool.

In [4]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_016mgJbYyUjpwgBE7iq14jTX', container=None, content=[TextBlock(citations=None, text='The best exercises for gaining leg muscle are compound movements that target multiple muscle groups:\n\n1. **Barbell Back Squat** - Often considered the king of leg exercises, squats work the quadriceps, hamstrings, glutes, and calves while also engaging your core. They allow you to lift heavy weight and progressively overload effectively.\n\n2. **Romanian Deadlifts** - Excellent for targeting the hamstrings and glutes, with significant lower back engagement. Great for posterior chain development.\n\n3. **Bulgarian Split Squats** - A unilateral exercise that builds leg strength while improving balance and correcting muscle imbalances between legs.\n\n4. **Leg Press** - Allows you to safely move heavy weight with less stress on your lower back, effectively targeting quads, hamstrings, and glutes.\n\n5. **Lunges** (walking or stationary) - Another great unilateral movement for overall leg 